In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

## Load model & tokenizer

In [2]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()


In [3]:
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9

Loading:  EleutherAI/gpt-j-6b


Some weights of the model checkpoint at EleutherAI/gpt-j-6b were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

## Load dataset and Compute task-conditioned mean activations

In [4]:
dataset = load_dataset('antonym', seed=0)
mean_activations = get_mean_head_activations(dataset, model, model_config, tokenizer)

## Compute function vector (FV)

In [5]:
FV, top_heads = compute_universal_function_vector(mean_activations, model, model_config, n_top_heads=10)

## Prompt Creation - ICL, Shuffled-Label, Zero-Shot, and Natural Text

In [10]:
# Sample ICL example pairs, and a test word
dataset = load_dataset('antonym')
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]

prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True)
sentence = create_prompt(prompt_data)
print("ICL prompt:\n", repr(sentence), '\n\n')

shuffled_prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
shuffled_sentence = create_prompt(shuffled_prompt_data)
print("Shuffled ICL Prompt:\n", repr(shuffled_sentence), '\n\n')

zeroshot_prompt_data = word_pairs_to_prompt_data({'input':[], 'output':[]}, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
zeroshot_sentence = create_prompt(zeroshot_prompt_data)
print("Zero-Shot Prompt:\n", repr(zeroshot_sentence))

ICL prompt:
 '<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 


Shuffled ICL Prompt:
 '<|endoftext|>Q: hardware\nA: democracy\n\nQ: fascism\nA: health\n\nQ: incompatible\nA: ignore\n\nQ: illness\nA: software\n\nQ: notice\nA: compatible\n\nQ: increase\nA:' 


Zero-Shot Prompt:
 '<|endoftext|>Q: increase\nA:'


## Evaluation

### Clean ICL Prompt

In [11]:
# Check model's ICL answer
clean_logits = sentence_eval(sentence, [test_pair['output']], model, tokenizer, compute_nll=False)

print("Input Sentence:", repr(sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("ICL Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')

Input Sentence: '<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

ICL Prompt Top K Vocab Probs:
 [(' decrease', 0.73723), (' reduce', 0.0777), (' increase', 0.03421), (' decline', 0.01579), (' decreased', 0.01035)] 



### Corrupted ICL Prompt

In [12]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: hardware\nA: democracy\n\nQ: fascism\nA: health\n\nQ: incompatible\nA: ignore\n\nQ: illness\nA: software\n\nQ: notice\nA: compatible\n\nQ: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' democracy', 0.01887), (' hardware', 0.01801), (' software', 0.01424), (' increase', 0.0101), (' health', 0.00927)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' decrease', 0.51291), (' reduce', 0.04482), (' decline', 0.02362), (' increase', 0.01525), (' improve', 0.00588)]


### Zero-Shot Prompt

In [13]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

Zero-Shot Top K Vocab Probs:
 [(' increase', 0.14926), (' yes', 0.02271), (' I', 0.02184), (' the', 0.02117), (' 1', 0.01421)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' decrease', 0.27559), (' increase', 0.17933), (' reduce', 0.03672), (' improve', 0.00965), ('\n', 0.0055)]


### Natural Text Prompt

In [14]:
sentence = f"The word \"{test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)


print("Input Sentence: ", repr(sentence))
print("GPT-J:" , repr(tokenizer.decode(co.squeeze())))
print("GPT-J+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "increase" means'
GPT-J: 'The word "increase" means "to make larger, to enlarge, to expand'
GPT-J+FV: 'The word "increase" means "decrease" in the Bible.\n' 



# Generalization

## Generalize to a new model: Gemma-2B

In [5]:
model_name = 'google/gemma-2b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9

Loading:  google/gemma-2b


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
dataset = load_dataset('antonym', seed=0)
mean_activations = get_mean_head_activations(dataset, model, model_config, tokenizer)

In [10]:
from src.utils.extract_utils import compute_function_vector
from src.compute_indirect_effect import compute_indirect_effect

indirect_effect = compute_indirect_effect(
    dataset,
    mean_activations,
    model=model,
    model_config=model_config,
    tokenizer=tokenizer,
    n_shots=10,
    n_trials=10,           
    last_token_only=True
)

FV, top_heads = compute_function_vector(
    mean_activations,
    indirect_effect,
    model,
    model_config,
    n_top_heads=10,
    token_class_idx=-1
)

100%|██████████| 10/10 [00:31<00:00,  3.16s/it]


In [ ]:
# Sample ICL example pairs, and a test word
dataset = load_dataset('antonym')
word_pairs = dataset['train'][:5]
test_pair = dataset['test'][21]

prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True)
sentence = create_prompt(prompt_data)
print("ICL prompt:\n", repr(sentence), '\n\n')

shuffled_prompt_data = word_pairs_to_prompt_data(word_pairs, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
shuffled_sentence = create_prompt(shuffled_prompt_data)
print("Shuffled ICL Prompt:\n", repr(shuffled_sentence), '\n\n')

zeroshot_prompt_data = word_pairs_to_prompt_data({'input':[], 'output':[]}, query_target_pair=test_pair, prepend_bos_token=True, shuffle_labels=True)
zeroshot_sentence = create_prompt(zeroshot_prompt_data)
print("Zero-Shot Prompt:\n", repr(zeroshot_sentence))

ICL prompt:
 '<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 


Shuffled ICL Prompt:
 '<|endoftext|>Q: hardware\nA: democracy\n\nQ: fascism\nA: software\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 


Zero-Shot Prompt:
 '<|endoftext|>Q: increase\nA:'


In [12]:
# Check model's ICL answer
clean_logits = sentence_eval(sentence, [test_pair['output']], model, tokenizer, compute_nll=False)

print("Input Sentence:", repr(sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("ICL Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')

Input Sentence: '<|endoftext|>Q: hardware\nA: software\n\nQ: fascism\nA: democracy\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

ICL Prompt Top K Vocab Probs:
 [(' decrease', 0.80273), (' reduce', 0.06189), (' decline', 0.01746), (' lower', 0.00576), (' diminish', 0.00477)] 



In [13]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(shuffled_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: hardware\nA: democracy\n\nQ: fascism\nA: software\n\nQ: incompatible\nA: compatible\n\nQ: illness\nA: health\n\nQ: notice\nA: ignore\n\nQ: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' decrease', 0.76025), (' reduce', 0.06647), (' decline', 0.01248), (' improve', 0.00474), (' lower', 0.00431)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' decrease', 0.83154), (' reduce', 0.05746), (' decline', 0.00742), (' increase', 0.0041), (' lower', 0.00309)]


In [14]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, [test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(test_pair['input'])}, Target: {repr(test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: increase\nA:' 

Input Query: 'increase', Target: 'decrease'

Zero-Shot Top K Vocab Probs:
 [(' decrease', 0.15491), (' <', 0.07202), (' increase', 0.05789), (' ', 0.03146), ('\n', 0.01292)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' decrease', 0.39404), (' increase', 0.07288), (' <', 0.02855), (' ', 0.02258), ('\n', 0.01552)]


In [15]:
sentence = f"The word \"{test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)


print("Input Sentence: ", repr(sentence))
print("GPT-J:" , repr(tokenizer.decode(co.squeeze())))
print("GPT-J+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "increase" means'
GPT-J: '<bos>The word "increase" means to grow or to become larger.\n\nThe word'
GPT-J+FV: '<bos>The word "increase" means "decrease."\n\nThe word "decrease" means' 



## Generalization to new data

In [6]:
custom_word_pairs = {
    "input": [
        "scarce",
        "opaque",
        "rigid",
        "ancient",
        "hostile"
    ],
    "output": [
        "abundant",
        "transparent",
        "flexible",
        "modern",
        "friendly"
    ]
}

custom_test_pair = {
    "input": "meticulous",
    "output": "careless"
}

prompt_data = word_pairs_to_prompt_data(
    custom_word_pairs,
    query_target_pair=custom_test_pair,
    prepend_bos_token=True
)
sentence = create_prompt(prompt_data)
print("ICL prompt:\n", repr(sentence), "\n")

shuffled_prompt_data = word_pairs_to_prompt_data(
    custom_word_pairs,
    query_target_pair=custom_test_pair,
    prepend_bos_token=True,
    shuffle_labels=True
)
shuffled_sentence = create_prompt(shuffled_prompt_data)
print("Shuffled ICL prompt:\n", repr(shuffled_sentence), "\n")

zeroshot_prompt_data = word_pairs_to_prompt_data(
    {"input": [], "output": []},
    query_target_pair=custom_test_pair,
    prepend_bos_token=True
)
zeroshot_sentence = create_prompt(zeroshot_prompt_data)
print("Zero-shot prompt:\n", repr(zeroshot_sentence))


ICL prompt:
 '<|endoftext|>Q: scarce\nA: abundant\n\nQ: opaque\nA: transparent\n\nQ: rigid\nA: flexible\n\nQ: ancient\nA: modern\n\nQ: hostile\nA: friendly\n\nQ: meticulous\nA:' 

Shuffled ICL prompt:
 '<|endoftext|>Q: scarce\nA: friendly\n\nQ: opaque\nA: flexible\n\nQ: rigid\nA: transparent\n\nQ: ancient\nA: abundant\n\nQ: hostile\nA: modern\n\nQ: meticulous\nA:' 

Zero-shot prompt:
 '<|endoftext|>Q: meticulous\nA:'


In [7]:
# Check model's ICL answer
clean_logits = sentence_eval(sentence, [custom_test_pair['output']], model, tokenizer, compute_nll=False)

print("Input Sentence:", repr(sentence), '\n')
print(f"Input Query: {repr(custom_test_pair['input'])}, Target: {repr(custom_test_pair['output'])}\n")
print("ICL Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')

Input Sentence: '<|endoftext|>Q: scarce\nA: abundant\n\nQ: opaque\nA: transparent\n\nQ: rigid\nA: flexible\n\nQ: ancient\nA: modern\n\nQ: hostile\nA: friendly\n\nQ: meticulous\nA:' 

Input Query: 'meticulous', Target: 'careless'

ICL Prompt Top K Vocab Probs:
 [(' careless', 0.52411), (' sloppy', 0.06768), (' casual', 0.0656), (' lax', 0.0437), (' care', 0.02249)] 



In [8]:
# Perform an intervention on the shuffled setting
clean_logits, interv_logits = function_vector_intervention(shuffled_sentence, [custom_test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(shuffled_sentence), '\n')
print(f"Input Query: {repr(custom_test_pair['input'])}, Target: {repr(custom_test_pair['output'])}\n")
print("Few-Shot-Shuffled Prompt Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Shuffled Prompt+FV Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: scarce\nA: friendly\n\nQ: opaque\nA: flexible\n\nQ: rigid\nA: transparent\n\nQ: ancient\nA: abundant\n\nQ: hostile\nA: modern\n\nQ: meticulous\nA:' 

Input Query: 'meticulous', Target: 'careless'

Few-Shot-Shuffled Prompt Top K Vocab Probs:
 [(' careless', 0.07832), (' efficient', 0.01919), (' meticulous', 0.01681), (' organized', 0.01495), (' lazy', 0.01372)] 

Shuffled Prompt+FV Top K Vocab Probs:
 [(' careless', 0.24812), (' casual', 0.0745), (' imp', 0.03384), (' sloppy', 0.02719), (' relaxed', 0.02656)]


In [9]:
# Intervention on the zero-shot prompt
clean_logits, interv_logits = function_vector_intervention(zeroshot_sentence, [custom_test_pair['output']], EDIT_LAYER, FV, model, model_config, tokenizer)

print("Input Sentence:", repr(zeroshot_sentence), '\n')
print(f"Input Query: {repr(custom_test_pair['input'])}, Target: {repr(custom_test_pair['output'])}\n")
print("Zero-Shot Top K Vocab Probs:\n", decode_to_vocab(clean_logits, tokenizer, k=5), '\n')
print("Zero-Shot+FV Vocab Top K Vocab Probs:\n", decode_to_vocab(interv_logits, tokenizer, k=5))

Input Sentence: '<|endoftext|>Q: meticulous\nA:' 

Input Query: 'meticulous', Target: 'careless'

Zero-Shot Top K Vocab Probs:
 [(' meticulous', 0.06057), (' careful', 0.02929), (' a', 0.01862), (' very', 0.01496), (' I', 0.01461)] 

Zero-Shot+FV Vocab Top K Vocab Probs:
 [(' meticulous', 0.06306), (' careful', 0.02755), (' thorough', 0.0267), (' careless', 0.01893), (' organized', 0.01148)]


In [10]:
sentence = f"The word \"{custom_test_pair['input']}\" means"
co, io = fv_intervention_natural_text(sentence, EDIT_LAYER, FV, model, model_config, tokenizer, max_new_tokens=10)


print("Input Sentence: ", repr(sentence))
print("GPT-J:" , repr(tokenizer.decode(co.squeeze())))
print("GPT-J+FV:", repr(tokenizer.decode(io.squeeze())), '\n')

Input Sentence:  'The word "meticulous" means'
GPT-J: 'The word "meticulous" means "careful, exact, and thorough." It'
GPT-J+FV: 'The word "meticulous" means "careful."\ncareless.\n\n' 

